In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style = "darkgrid")

from utils.helper import *

#### V1

In [ ]:
v1 = pd.read_parquet("../data/clean_insurance_claims.parquet")

# remove columns that are not useful for modelling, high cardinality, ID features, and fraud class
v1 = v1.drop(columns = ["policy_number", "insured_zip", "incident_location", "fraud_reported", "auto_model", 
                        "auto_year", "policy_bind_date", "policy_state"])

# remove columns that are highly correlated with the target 
v1 = v1.drop(columns = ["injury_claim", "property_claim", "vehicle_claim"])

# 'age' is highly correlated with 'months_as_customer'
v1 = v1.drop(columns = ["age"])

In [ ]:
# timestamps are split to the day component which allows for temporal patterns to be captured
v1["incident_date_day"] = v1["incident_date"].dt.day

v1 = v1.drop(columns = ["incident_date"])

In [ ]:
v1.to_parquet("../data/claims_model_dataset_v1.parquet")

#### V2

In [ ]:
v2 = pd.read_parquet("../data/clean_insurance_claims.parquet")

v2 = v2.drop(columns = ["injury_claim", "property_claim", "vehicle_claim", "policy_number", "insured_zip", "age", 
                        "incident_location", "fraud_reported", "policy_bind_date", "policy_state"])

v2["incident_date_day"] = v2["incident_date"].dt.day
v2 = v2.drop(columns = ["incident_date"])

In [ ]:
# currently, the feature 'auto_model' is not being used due to its high cardinality
# however, this feature is useful in understanding the type of car involved in the incident
# to use this feature, I will classify each make and model into vehicle classes, e.g. sedan, hatchback...
# when mapping vehicle classes, the end result 5 classes: sedan, suv, truck, hatchback, coupe
# the problem with this split, is that hatchback and coupe have 48 and 12 entires in total
# this means that coupe will be treated as noise and not give the models real patterns
# to combat this, coupe and hatchback will be merged into other
vehicle_classes = {
    "92x": "other",
    "E400": "sedan",
    "RAM": "truck",
    "Tahoe": "suv",
    "RSX": "other",
    "95": "sedan",
    "Pathfinder": "suv",
    "A5": "sedan", 
    "Camry": "sedan", 
    "F150": "truck",
    "A3": "sedan",
    "Highlander": "suv",
    "Neon": "sedan",
    "MDX": "suv",
    "Maxima": "sedan",
    "Legacy": "sedan",
    "TL": "sedan",
    "Impreza": "other",
    "Forrestor": "suv", 
    "Escape": "suv",
    "Corolla": "other",
    "3 Series": "sedan",
    "C300": "sedan",
    "Wrangler": "suv", 
    "M5": "sedan",
    "X5": "suv",
    "Civic": "sedan",
    "Passat": "sedan",
    "Silverado": "truck",
    "CRV": "suv",
    "93": "sedan",
    "Accord": "sedan",
    "X6": "suv",
    "Malibu": "sedan",
    "Fusion": "sedan",
    "Jetta": "sedan",
    "ML350": "suv",
    "Ultima": "sedan",
    "Grand Cherokee": "suv"
}

v2["vehicle_class"] = v2["auto_model"].map(vehicle_classes)
v2 = v2.drop(columns = ["auto_model"])

v2["vehicle_class"].value_counts()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (10, 4), constrained_layout = True)
axes = axes.flatten() 
plt.suptitle("Distribution of Vehicle Class", fontsize = 14, fontweight = "semibold")

sns.boxplot(data = v2, x = "vehicle_class", y = "total_claim_amount", ax = axes[0])
sns.histplot(data = v2, x = "total_claim_amount", hue = "vehicle_class", ax = axes[1], kde = True)

fig.savefig("../figures/claims/feature_engineering/claims_vehicle_class_feature.png", dpi = 300)
plt.show()

In [ ]:
v2.to_parquet("../data/claims_model_dataset_v2.parquet")

#### V3

In [ ]:
v3 = pd.read_parquet("../data/clean_insurance_claims.parquet")

v3 = v3.drop(columns = ["injury_claim", "property_claim", "vehicle_claim", "policy_number", "insured_zip", "age", 
                        "incident_location", "fraud_reported", "policy_bind_date", "policy_state"])

v3["vehicle_class"] = v3["auto_model"].map(vehicle_classes)
v3["incident_date_day"] = v3["incident_date"].dt.day
v3 = v3.drop(columns = ["incident_date", "auto_model"])

In [ ]:
# 'incident_hour_of_the_day' is an interesting feature on its own, but it is possible to split it up into time periods
# these time periods should reduce noise in the data and improve model performance
v3["incident_time_period"] = v3["incident_hour_of_the_day"].apply(time_period)
v3 = v3.drop(columns = ["incident_hour_of_the_day"])

v3["incident_time_period"].value_counts()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (10, 4), constrained_layout = True)
axes = axes.flatten()
plt.suptitle("Distribution of Incident Time Period", fontsize = 14, fontweight = "semibold")

sns.boxplot(data = v3, x = "incident_time_period", y = "total_claim_amount", ax = axes[0])
sns.histplot(data = v3, x = "total_claim_amount", hue = "incident_time_period", ax = axes[1], kde = True)

fig.savefig("../figures/claims/feature_engineering/claims_incident_time_period_feature.png", dpi = 300)
plt.show()

In [ ]:
v3.to_parquet("../data/claims_model_dataset_v3.parquet")